# 第1回 ガイダンス：データ分析ツールの使い分け

**データ分析（da）／ 情報工学部 情報工学科 2年次 前期 ／ 担当: 後藤 弘光**

---

本科目のキーメッセージ：**「描いてから、書く」**

データ分析は、いきなりコードを書くものではありません。まず分析の骨組み（手順）を日本語で設計し、それを一つずつ Python コードに翻訳していきます。生成 AI も、骨組みが明確なほど正確なコードを返してくれます。本科目ではこの「設計 → 翻訳」の習慣を 15 週かけて身につけていきます。

## 1. 要点まとめ

### 1.1 なぜ Python か

Excel は優れた表計算ソフトですが、データ分析の実務では次のような課題があります。

| 観点 | Excel | Python |
| :-- | :-- | :-- |
| 再現性 | 手作業の履歴が残りにくい | コードとして手順が残る |
| スケール | 数十万行で重くなる | 数千万行も実用的 |
| 自動化 | マクロが必要 | スクリプトで自然に自動化 |
| 分析手法 | 基本統計まで | 回帰・検定・機械学習まで |
| 共有 | ファイルごと配布 | notebook で分析過程ごと共有 |

### 1.2 Pandas とは

Python でデータを表形式（DataFrame）として扱うためのライブラリです。`pd.read_csv()` で CSV を読み込み、`head()` で先頭を確認するのが分析の第一歩になります。

### 1.3 本科目の学び方

毎回の授業は以下のフローで進めます。

1. **アプローチ選択クイズ**（Moodle、10分）— 分析アプローチの選択トレーニング
2. **ストーリー構築**（20分）— 背景から分析ストーリーを議論、教員が iPad で板書
3. **要点解説＋骨組み作成**（30分）— 本時の要点を解説し、コードの骨組み（コメント行）を全員で確定
4. **伴走型実装**（20分）— 骨組みを Python コードに翻訳
5. **サマリー入力**（Moodle、10分）— 演習結果を Moodle 小テストで記入・提出

このフローを 15 回繰り返すことで、分析の思考手順が自然に身についていきます。

クイズでは **4つの統計分析手法**（回帰・検定・クラスタ・分散分析）のどれが適切かを問う問題が毎回出題されます。まずは次節で、それぞれがどんな分析なのかを「動くコード」で体験してみましょう。

### 1.4 Python でできる分析のプレビュー

本科目の後半（第11-14回）で詳しく学ぶ **4つの統計分析**を、先に結果だけ体験してみます。今はコードを理解する必要はありません。「こんなことができるのか」というイメージを掴むことが目的です。

共通データとして、レストランのチップ実績データ（`seaborn` の `tips` データセット：会計額・チップ額・曜日・時間帯・人数など244件）を使用します。

In [ ]:
# 共通準備：ライブラリとデータの読み込み
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tips = sns.load_dataset('tips')
tips.head()

#### ① 回帰分析（第11回で詳しく）

**「Aが増えるとBはどれくらい増えるか？」を数式にする分析です。**

ここでは「会計額 (`total_bill`) が増えると、チップ (`tip`) はどれくらい増えるか？」を調べてみます。

In [ ]:
from sklearn.linear_model import LinearRegression

X = tips[['total_bill']].values
y = tips['tip'].values
model = LinearRegression().fit(X, y)

print(f'回帰式: tip = {model.coef_[0]:.3f} × total_bill + {model.intercept_:.3f}')

sns.regplot(x='total_bill', y='tip', data=tips, line_kws={'color': 'red'})
plt.title('① 回帰分析：会計額とチップの関係')
plt.show()

#### ② 統計検定（第12回で詳しく）

**「2つのグループに本当に差があるか？」を確率で判定する分析です。**

ここでは「ランチとディナーでチップ額に有意な差があるか？」を調べてみます。

In [ ]:
from scipy import stats

lunch = tips[tips['time'] == 'Lunch']['tip']
dinner = tips[tips['time'] == 'Dinner']['tip']
t, p = stats.ttest_ind(lunch, dinner)

print(f'ランチの平均チップ: {lunch.mean():.2f}, ディナーの平均チップ: {dinner.mean():.2f}')
print(f't値 = {t:.3f}, p値 = {p:.4f}')
print('→ p < 0.05 なら「差は偶然ではない」と言えます')

sns.boxplot(x='time', y='tip', data=tips)
plt.title('② 統計検定：ランチ vs ディナーのチップ比較')
plt.show()

#### ③ クラスタ分析（第13回で詳しく）

**「データを似たもの同士のグループに自動で分ける」分析です。**

ここでは「会計額と人数の2軸で、客層を3つのグループに自動分類」してみます。

In [ ]:
from sklearn.cluster import KMeans

X = tips[['total_bill', 'size']].values
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
tips_vis = tips.copy()
tips_vis['cluster'] = km.labels_

sns.scatterplot(x='total_bill', y='size', hue='cluster', data=tips_vis, palette='Set1')
plt.title('③ クラスタ分析：会計額×人数で3グループに自動分類')
plt.show()

#### ④ 分散分析（第14回で詳しく）

**「3つ以上のグループの平均に差があるか？」を判定する分析です。**

ここでは「曜日（木・金・土・日）でチップ額に差があるか？」を調べてみます。

In [ ]:
groups = [tips[tips['day'] == d]['tip'].values for d in ['Thur', 'Fri', 'Sat', 'Sun']]
f, p = stats.f_oneway(*groups)

print(f'F値 = {f:.3f}, p値 = {p:.4f}')
print('→ p < 0.05 なら「少なくとも1つの曜日の平均は他と異なる」と言えます')

sns.boxplot(x='day', y='tip', data=tips, order=['Thur', 'Fri', 'Sat', 'Sun'])
plt.title('④ 分散分析：曜日別チップの比較')
plt.show()

以上の4つが、本科目の後半で「自分の手で」書けるようになる分析です。今は「どんな問いにはどの手法が向くか」の直感だけを覚えておきましょう。

| 手法 | 向いている問い |
| :-- | :-- |
| 回帰分析 | Aが増えるとBはどう変わる？（量 vs 量） |
| 統計検定 | 2つのグループに差がある？ |
| クラスタ分析 | データを似たもの同士に自動で分けたい |
| 分散分析 | 3つ以上のグループに差がある？ |

---

## 2. 演習

### 2.1 背景

> これまで売上分析を Excel で行ってきましたが、データ量が増えて動作が重く、そもそもどんな項目がどのくらい入っているのかをすぐに確認することもできなくなってきました。

#### 問い

Excel では扱いきれなくなってきたデータ量（**スケール**）と、目視確認に頼っていた作業（**再現性**）の課題を、Python に移行することでどのように解決できるか？

#### 仮説

CSV を Python で読み込み、データの中身（行数・列名・先頭数行）をコードで確認できれば、Excel で「目で見る」作業をコードに置き換える第一歩になるはずです。

### 2.2 分析

以下のコードセルに、まず分析の骨組みをコメント行として書き出し、その下に一つずつ Python コードを翻訳していきます。

In [ ]:
# ============================================================
# 設計 → 実装
# ------------------------------------------------------------
# データ分析は「いきなりコードを書く」ものではありません。
# まず日本語で分析の骨組み（手順）を書き出し、
# それを一つずつ Python コードに翻訳していきます。
#
# このセルには授業中に以下の流れで記入します:
#   1. 骨組みコメントを書く（# 1. …, # 2. …, # 3. …）
#      ← 「何をするか」を日本語で明文化
#   2. 各コメントの下にコードを追記する
#      ← 日本語の手順を Python に「翻訳」
#
# 生成 AI も、骨組みが明確なほど正確なコードを返してくれます。
# この「描いてから書く」習慣を本科目で身につけましょう。
# ============================================================

# 1.

# 2.

# 3.

# 4.

# 5.


---

### 2.3 サマリー

以下の穴埋め項目を、演習で実行したコードの出力結果をもとに記入してください。

#### 分析結果の確認

- 読み込んだ CSV ファイルの行数は \_\_\_\_ 行でした
- 列数は \_\_\_\_ 列でした
- 列名は \_\_\_\_ でした

#### 結論

上記の確認により、Excel で「目視確認」していた作業を、Python の \_\_\_\_（CSV 読込）、\_\_\_\_（形状確認）、\_\_\_\_（列名確認）という数行のコードで置き換えられることを確認できました。